## Imports

In [1]:
import torch
from torch import nn
import requests

/Users/bobr_oleje/miniforge3/envs/pure-torch/lib/python3.11/site-packages/torch/_subclasses/functional_tensor.py:283: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [2]:
import sys
import os
import importlib
sys.path.append(os.path.abspath('..'))
sys.path.append(os.path.abspath('../src'))
sys.path.append(os.path.abspath('../models'))

In [3]:
from src.models.GPT_architecture import GPT, GPT_RoPE

## With no positional encodings

In [10]:
# ---- hyperparams ----
EMBEDDING_DIM  = 128
NUM_HEADS      = 4
HIDDEN_DIM     = 512
NUM_LAYERS     = 4
CONTEXT_LEN    = 64
BATCH_SIZE     = 64
LR             = 3e-4
STEPS          = 10_000
EVAL_INTERVAL  = 500
EVAL_STEPS     = 50
DEVICE         = 'mps' if torch.mps.is_available() else "cpu"
print(f"{DEVICE=}")

# ---- data ----
def get_data():
    path = "shakespeare.txt"
    if not os.path.exists(path):
        url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
        print("Downloading TinyShakespeare...")
        r = requests.get(url)
        with open(path, "w") as f:
            f.write(r.text)
    with open(path, "r") as f:
        text = f.read()
    return text

text = get_data()
chars = sorted(set(text))
VOCAB_SIZE = len(chars)
print(f"Vocab size: {VOCAB_SIZE}, dataset size: {len(text):,} chars")

stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for i, c in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: "".join(itos[i] for i in l)

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data   = data[n:]

def get_batch(split):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - CONTEXT_LEN, (BATCH_SIZE,))
    x = torch.stack([d[i:i+CONTEXT_LEN] for i in ix])
    y = torch.stack([d[i+1:i+CONTEXT_LEN+1] for i in ix])
    return x.to(DEVICE), y.to(DEVICE)

# ---- training ----
model = GPT(
    vocab_size=VOCAB_SIZE,
    embedding_dim=EMBEDDING_DIM,
    num_heads=NUM_HEADS,
    hidden_dim=HIDDEN_DIM,
    context_len=CONTEXT_LEN,
    num_transformer_blocks=NUM_LAYERS,
    loss_fn=nn.CrossEntropyLoss(),
    positional_encoding=nn.Embedding(CONTEXT_LEN, EMBEDDING_DIM)
).to(DEVICE)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

@torch.no_grad()
def estimate_loss():
    model.eval()
    out = {}
    for split in ["train", "val"]:
        losses = []
        for _ in range(EVAL_STEPS):
            x, y = get_batch(split)
            _, loss = model(x, y)
            losses.append(loss.item())
        out[split] = sum(losses) / len(losses)
    model.train()
    return out

for step in range(STEPS):
    if step % EVAL_INTERVAL == 0:
        losses = estimate_loss()
        print(f"step {step:4d} | train loss {losses['train']:.4f} | val loss {losses['val']:.4f}")

    x, y = get_batch("train")
    _, loss = model(x, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

print("\n--- sample ---")
ctx = torch.zeros((1, 1), dtype=torch.long, device=DEVICE)
print(decode(model.generate(ctx, max_new_tokens=500)[0].tolist()))

DEVICE='mps'
Vocab size: 65, dataset size: 1,115,394 chars
Parameters: 818,241
step    0 | train loss 4.3832 | val loss 4.3853
step  500 | train loss 2.1501 | val loss 2.1758
step 1000 | train loss 1.8863 | val loss 1.9735
step 1500 | train loss 1.7310 | val loss 1.8697
step 2000 | train loss 1.6352 | val loss 1.8039
step 2500 | train loss 1.5638 | val loss 1.7476
step 3000 | train loss 1.5279 | val loss 1.7070
step 3500 | train loss 1.4942 | val loss 1.6893
step 4000 | train loss 1.4631 | val loss 1.6497
step 4500 | train loss 1.4325 | val loss 1.6417
step 5000 | train loss 1.4206 | val loss 1.6168
step 5500 | train loss 1.3994 | val loss 1.6009
step 6000 | train loss 1.3930 | val loss 1.6112
step 6500 | train loss 1.3711 | val loss 1.5930
step 7000 | train loss 1.3597 | val loss 1.5738
step 7500 | train loss 1.3384 | val loss 1.5824
step 8000 | train loss 1.3348 | val loss 1.5646
step 8500 | train loss 1.3296 | val loss 1.5754
step 9000 | train loss 1.3170 | val loss 1.5667
step 9500

## With positional encodings

In [ ]:
# ---- hyperparams ----
EMBEDDING_DIM  = 128
NUM_HEADS      = 4
HIDDEN_DIM     = 512
NUM_LAYERS     = 4
CONTEXT_LEN    = 64
BATCH_SIZE     = 64
LR             = 3e-4
STEPS          = 10_000
EVAL_INTERVAL  = 500
EVAL_STEPS     = 50
DEVICE         = 'mps' if torch.mps.is_available() else "cpu"
print(f'{DEVICE=}')

# ---- data ----
def get_data():
    path = "shakespeare.txt"
    if not os.path.exists(path):
        url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
        print("Downloading TinyShakespeare...")
        r = requests.get(url)
        with open(path, "w") as f:
            f.write(r.text)
    with open(path, "r") as f:
        text = f.read()
    return text

text = get_data()
chars = sorted(set(text))
VOCAB_SIZE = len(chars)
print(f"Vocab size: {VOCAB_SIZE}, dataset size: {len(text):,} chars")

stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for i, c in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: "".join(itos[i] for i in l)

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data   = data[n:]

def get_batch(split):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - CONTEXT_LEN, (BATCH_SIZE,))
    x = torch.stack([d[i:i+CONTEXT_LEN] for i in ix])
    y = torch.stack([d[i+1:i+CONTEXT_LEN+1] for i in ix])
    return x.to(DEVICE), y.to(DEVICE)

# ---- training ----
model = GPT_RoPE(
    vocab_size=VOCAB_SIZE,
    embedding_dim=EMBEDDING_DIM,
    num_heads=NUM_HEADS,
    hidden_dim=HIDDEN_DIM,
    context_len=CONTEXT_LEN,
    num_transformer_blocks=NUM_LAYERS,
    loss_fn=nn.CrossEntropyLoss()
).to(DEVICE)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

@torch.no_grad()
def estimate_loss():
    model.eval()
    out = {}
    for split in ["train", "val"]:
        losses = []
        for _ in range(EVAL_STEPS):
            x, y = get_batch(split)
            _, loss = model(x, y)
            losses.append(loss.item())
        out[split] = sum(losses) / len(losses)
    model.train()
    return out

for step in range(STEPS):
    if step % EVAL_INTERVAL == 0:
        losses = estimate_loss()
        print(f"step {step:4d} | train loss {losses['train']:.4f} | val loss {losses['val']:.4f}")

    x, y = get_batch("train")
    _, loss = model(x, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

print("\n--- sample ---")
ctx = torch.zeros((1, 1), dtype=torch.long, device=DEVICE)
print(decode(model.generate(ctx, max_new_tokens=500)[0].tolist()))

DEVICE='mps'
Vocab size: 65, dataset size: 1,115,394 chars
Parameters: 810,049
step    0 | train loss 4.3066 | val loss 4.3087
step  500 | train loss 1.9046 | val loss 1.9850
step 1000 | train loss 1.7139 | val loss 1.8613
step 1500 | train loss 1.6209 | val loss 1.7928
step 2000 | train loss 1.5510 | val loss 1.7445
step 2500 | train loss 1.5099 | val loss 1.7051
step 3000 | train loss 1.4763 | val loss 1.6763
step 3500 | train loss 1.4495 | val loss 1.6564
